# 🍀 Plant Disease Final Evaluation System

This notebook serves as the final prediction interface for the **Advanced Meta-Ensemble Plant Disease Detection** project. It leverages the trained model weights and an enhanced metadata JSON to provide high-precision diagnosis and actionable remedies.

### 🧬 Core Components:
1. **Trained Model**: `best_plant_disease_model.pth` (Dual-Engine Meta-Ensemble)
2. **Metadata DB**: `class_names_enhanced.json` (Reasons & Solutions)
3. **Inference Logic**: Preprocessing + Probabilistic Fusion

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import json
import matplotlib.pyplot as plt
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 🏗️ Standalone Model Architecture
Defining the Meta-Ensemble (MobileNetV2 + DenseNet121) to ensure the notebook remains portable.

In [ ]:
class PlantDiseaseEnsemble(nn.Module):
    def __init__(self, num_classes=38, weight_A=0.635, weight_B=0.365):
        super(PlantDiseaseEnsemble, self).__init__()
        
        # Model A: MobileNetV2
        self.model_A = models.mobilenet_v2(weights=None)
        in_A = self.model_A.classifier[1].in_features
        self.model_A.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_A, num_classes))
        
        # Model B: DenseNet121
        self.model_B = models.densenet121(weights=None)
        in_B = self.model_B.classifier.in_features
        self.model_B.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_B, num_classes))
        
        self.weight_A = weight_A
        self.weight_B = weight_B
        
    def forward(self, x):
        out_A = self.model_A(x)
        out_B = self.model_B(x)
        return (self.weight_A * out_A) + (self.weight_B * out_B)

### 🧪 Prediction Interface
Loading weights and metadata for real-time inference.

In [ ]:
PTH_PATH = 'best_plant_disease_model.pth'
JSON_PATH = 'class_names_enhanced.json'

# Initialize Model
model = PlantDiseaseEnsemble(num_classes=38)

# Load Weights
if os.path.exists(PTH_PATH):
    checkpoint = torch.load(PTH_PATH, map_location=device)
    state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    model.to(device).eval()
    print("✅ Model Weights Loaded Successfully")
else:
    print(f"❌ Error: {PTH_PATH} not found.")

# Load Metadata
if os.path.exists(JSON_PATH):
    with open(JSON_PATH, 'r') as f:
        metadata = json.load(f)
    print("✅ Enhanced Metadata Loaded Successfully")
else:
    print(f"❌ Error: {JSON_PATH} not found.")

### 📸 Diagnostic Pipeline

In [ ]:
def diagnose_plant(image_path):
    # 1. Load & Transform
    img = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    tensor = transform(img).unsqueeze(0).to(device)
    
    # 2. Predict
    with torch.no_grad():
        logits = model(tensor)
        probs = F.softmax(logits, dim=1)[0]
    
    conf, idx = torch.max(probs, dim=0)
    class_names = sorted(list(metadata.keys()))
    prediction = class_names[idx.item()]
    
    # 3. Display
    plt.imshow(img)
    plt.title(f"Diagnosis: {prediction} ({conf.item()*100:.2f}%)")
    plt.axis('off')
    plt.show()
    
    # 4. Detailed Report
    info = metadata.get(prediction, {})
    print("\n" + "="*50)
    print(f"🌿 FINAL DIAGNOSIS: {prediction}")
    print("="*50)
    print(f"📍 Plant  : {info.get('plant', 'N/A')}")
    print(f"🔴 Disease: {info.get('disease', 'N/A')}")
    
    print("\n🔍 REASONS:")
    for r in info.get('reasons', ['No data available.']):
        print(f"   - {r}")
        
    print("\n🛠️ SOLUTIONS:")
    for s in info.get('solutions', ['No data available.']):
        print(f"   ✅ {s}")
    print("="*50 + "\n")

### 🚀 Run Inference
Replace the path below with your own leaf image to get the diagnosis.

In [ ]:
# Example Usage:
IMAGE_PATH = 'sample_leaf.jpg' # Update with your image path
if os.path.exists(IMAGE_PATH):
    diagnose_plant(IMAGE_PATH)
else:
    print("⚠️ Provide a valid IMAGE_PATH to run the diagnosis.")